In [91]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
%matplotlib inline
from tensorflow import keras
import seaborn as sns 

In [75]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [76]:
train.drop(columns=['id', 'Race', 'Year', 'Driver'],inplace=True,  axis = 1)

In [77]:
train.head()

,Compound,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,HARD,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,HARD,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,HARD,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,MEDIUM,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,HARD,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [78]:
print(train.isnull().sum())

Compound                  0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64


In [79]:
from sklearn.model_selection import train_test_split
X = train.drop('PitNextLap', axis = 1)
y =train['PitNextLap']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)
X_train.shape, X_test.shape

((351312, 11), (87828, 11))

In [80]:
X_train.columns

Index(['Compound', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change'],
      dtype='object')

In [81]:
# Create Column Transformer with 3 types of transformers
cat_features = X.select_dtypes(include="object").columns
num_features = X.select_dtypes(exclude="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [82]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout

model = Sequential()

model.add(Dense(64, activation= 'relu', input_shape = (X_train.shape[1],)))
model.add(Dropout(.2))


model.add(Dense(32, activation= 'relu'))
model.add(Dropout(.2))

model.add(Dense(1, activation ='sigmoid'))

C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [83]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # binary target
    metrics=['accuracy']
)

In [85]:
history = model.fit(X_train, y_train,
                    validation_data = (X_test, y_test),
                    epochs = 30,
                    batch_size = 128)

Epoch 1/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.8529 - loss: 0.3375 - val_accuracy: 0.8718 - val_loss: 0.2941
Epoch 2/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.8713 - loss: 0.3011 - val_accuracy: 0.8747 - val_loss: 0.2856
Epoch 3/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 19s 7ms/step - accuracy: 0.8743 - loss: 0.2927 - val_accuracy: 0.8794 - val_loss: 0.2790
Epoch 4/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.8759 - loss: 0.2886 - val_accuracy: 0.8798 - val_loss: 0.2742
Epoch 5/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.8774 - loss: 0.2842 - val_accuracy: 0.8810 - val_loss: 0.2719
Epoch 6/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8782 - loss: 0.2813 - val_accuracy: 0.8813 - val_loss: 0.2682
Epoch 7/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8782 - loss: 0.2790 - val_accuracy: 0.8816 - val_loss: 0.2662
Epoch 8/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8783 - loss: 0.2772 -

In [86]:
loss, acc = model.evaluate(X_test, y_test)
print("Validation Accuracy:", acc)

2745/2745 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.8835 - loss: 0.2568
Validation Accuracy: 0.8835451006889343


In [ ]:
X_test_scaled = numeric_transformer.transform(X_test)
y_pred_prob = model.predict(X_test_scaled)

# Convert probabilities to binary
y_pred = (y_pred_prob > 0.5).astype(int)

# Prepare submission
if len(y_pred) != len(test):
    test_features = test.drop(columns=["id", "Race", "Year", "Driver"], errors="ignore")
    test_processed = preprocessor.transform(test_features)
    y_pred_prob = model.predict(test_processed)
    y_pred = (y_pred_prob > 0.5).astype(int)

submission = pd.DataFrame({
    "id": test["id"].values,
    "PitNextLap": y_pred.flatten()
})
submission.to_csv("submission.csv", index=False)

2745/2745 ━━━━━━━━━━━━━━━━━━━━ 2s 808us/step
5881/5881 ━━━━━━━━━━━━━━━━━━━━ 5s 778us/step
